In [1]:
import pandas as pd
import sqlite3
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [4]:
conn = sqlite3.connect("basketball_reference.db")

#data frames
tables = [
    "seasons", "teams", "players", "player_position", 
    "coaches", "awards", "award_season", "player_stats", "coach_stats"
]
dfs = {}
for table in tables:
    dfs[table] = pd.read_sql_query(f"SELECT * FROM {table};", conn)

In [5]:
def cleaning_dataframe(df, name, pk_cols, replacement="median", threshold=0.05):
   df_cleaned = df.copy()
    
   if pk_cols:
        df_cleaned = df_cleaned.drop_duplicates(subset=pk_cols)
    
   total_rows = len(df_cleaned)
   if total_rows == 0:
        return df_cleaned
        
   numeric_cols = []

   for col in df_cleaned.columns:
      dtype_name = df_cleaned[col].dtype.name

      if 'int' in dtype_name or 'float' in dtype_name:
         numeric_cols.append(col)
    
   for col in df_cleaned.columns:
        null_count = df_cleaned[col].isnull().sum()
        if null_count == 0:
            continue
            
        null_ratio = null_count / total_rows
        
        if null_ratio <= threshold:
            df_cleaned = df_cleaned.dropna(subset=[col])
        else:
            if col in numeric_cols:
                if replacement == "mean":
                    fill_value = df_cleaned[col].mean()
                elif replacement == "median":
                    fill_value = df_cleaned[col].median()
                else:
                    fill_value = 0
                df_cleaned[col] = df_cleaned[col].fillna(fill_value)
            else:
                df_cleaned[col] = df_cleaned[col].fillna("Unknown")
                
   return df_cleaned

In [7]:
dfs_clean = {}

dfs_clean["seasons"] = cleaning_dataframe(dfs["seasons"], "seasons", ["season_id"])
dfs_clean["teams"] = cleaning_dataframe(dfs["teams"], "teams", ["team_id"])
dfs_clean["players"] = cleaning_dataframe(dfs["players"], "players", ["player_id"])
dfs_clean["player_position"] = cleaning_dataframe(dfs["player_position"], "player_position", ["id"])
dfs_clean["coaches"] = cleaning_dataframe(dfs["coaches"], "coaches", ["coach_id"])
dfs_clean["awards"] = cleaning_dataframe(dfs["awards"], "awards", ["award_id"])
dfs_clean["award_season"] = cleaning_dataframe(dfs["award_season"], "award_season", ["id"])
dfs_clean["player_stats"] = cleaning_dataframe(dfs["player_stats"], "player_stats", pk_cols=["id"])
dfs_clean["coach_stats"] = cleaning_dataframe(dfs["coach_stats"], "coach_stats", ["id"])

